<a href="https://colab.research.google.com/github/AyaanMahajan/econ3916/blob/main/Econ_3916_PS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2: The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5 to 8 · 100 points · your role: Data Analyst, Growth team

This notebook follows the problem set in rubric order. I kept every heading and worked through the phases in order so each result builds on the previous one.


## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import TTestIndPower
from sklearn.linear_model import LogisticRegression
import warnings

rng = np.random.default_rng(42)
pd.set_option("display.precision", 3)


## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [2]:
# Parameters fixed by the problem
N_USERS = 50_000
BASE_BEFORE = 3.432
BASE_AFTER = 3.532


def simulate_population(base, n=N_USERS, rng=rng):
    """Create one simulated user population."""
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)
    engagement_z = (
        (engagement_minutes - engagement_minutes.mean())
        / engagement_minutes.std()
    )

    had_support_ticket = rng.random(n) < 0.22

    s = (
        base
        + 0.35 * engagement_z
        - 0.60 * had_support_ticket
        + rng.normal(0, 0.75, size=n)
    )

    satisfaction = np.clip(np.rint(s), 1, 5)

    return pd.DataFrame({
        "engagement_minutes": engagement_minutes,
        "engagement_z": engagement_z,
        "had_support_ticket": had_support_ticket,
        "s": s,
        "satisfaction": satisfaction,
    })


before = simulate_population(BASE_BEFORE)
after = simulate_population(BASE_AFTER)

true_before = before["satisfaction"].mean()
true_after = after["satisfaction"].mean()
true_change = true_after - true_before

print(f"True mean before: {true_before:.3f}")
print(f"True mean after:  {true_after:.3f}")
print(f"True change:      {true_change:.3f} points")


True mean before: 3.291
True mean after:  3.391
True change:      0.100 points


The true population mean is about 3.29 before and 3.39 after so the real improvement is basically 0.10 points. These are the numbers for all users in the simulated population and they are the benchmark I use later when I check whether the survey is telling the truth.


### Step 1.2: Impose the response mechanism (15 pts)

In [3]:
def respond(df, a, b, rng=rng):
    """Return one True or False response decision for every user."""
    z = (
        a
        + b * (df["s"] - 3.3)
        + 0.30 * df["engagement_z"]
        - 0.25 * df["had_support_ticket"]
    )
    p = 1 / (1 + np.exp(-z))
    return rng.random(len(df)) < p


responded_before = respond(before, a=-1.0, b=0.45)
responded_after = respond(after, a=-2.75, b=2.00)

pre_survey = before.loc[responded_before, "satisfaction"].to_numpy()
post_survey = after.loc[responded_after, "satisfaction"].to_numpy()

observed_before = pre_survey.mean()
observed_after = post_survey.mean()
observed_change = observed_after - observed_before

response_rate_before = responded_before.mean()
response_rate_after = responded_after.mean()

print(f"Observed mean before: {observed_before:.3f}")
print(f"Observed mean after:  {observed_after:.3f}")
print(f"Observed change:      {observed_change:.3f} points")
print(f"Response rate before: {response_rate_before:.1%}")
print(f"Response rate after:  {response_rate_after:.1%}")


Observed mean before: 3.592
Observed mean after:  4.309
Observed change:      0.717 points
Response rate before: 27.5%
Response rate after:  15.3%


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

The checkpoint matches. The survey mean is about 3.59 before and 4.31 after even though the true improvement is only about 0.10 points and the response rate actually falls from about 27.5 percent to 15.3 percent.

This is selection bias from nonresponse. Satisfaction affects whether someone answers and that relationship gets much stronger after the redesign so the respondents are not a random picture of all users. A bigger survey would not fix this because getting more responses from the same biased response process just makes the wrong estimate more precise.


## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [4]:
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Resample x with replacement and calculate one statistic each time."""
    boot_stats = []
    for _ in range(B):
        sample = rng.choice(x, size=len(x), replace=True)
        boot_stats.append(stat(sample))
    return np.array(boot_stats)


# Manual bootstrap first
mean_boot = bootstrap_by_hand(post_survey, np.mean)
median_boot = bootstrap_by_hand(post_survey, np.median)

mean_manual = np.percentile(mean_boot, [2.5, 97.5])
median_manual = np.percentile(median_boot, [2.5, 97.5])

# Same intervals with scipy
mean_percentile = stats.bootstrap(
    (post_survey,),
    np.mean,
    n_resamples=2000,
    method="percentile",
    random_state=123,
    vectorized=True,
    batch=100,
).confidence_interval

mean_bca = stats.bootstrap(
    (post_survey,),
    np.mean,
    n_resamples=2000,
    method="BCa",
    random_state=123,
    vectorized=True,
    batch=100,
).confidence_interval

median_percentile = stats.bootstrap(
    (post_survey,),
    np.median,
    n_resamples=2000,
    method="percentile",
    random_state=123,
    vectorized=True,
    batch=100,
).confidence_interval

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    median_bca = stats.bootstrap(
        (post_survey,),
        np.median,
        n_resamples=2000,
        method="BCa",
        random_state=123,
        vectorized=True,
        batch=100,
    ).confidence_interval

print("Post redesign respondent sample")
print(f"Mean:   {post_survey.mean():.3f}")
print(f"Median: {np.median(post_survey):.1f}")
print()
print(f"Manual mean percentile CI:   [{mean_manual[0]:.3f}, {mean_manual[1]:.3f}]")
print(f"Scipy mean percentile CI:    [{mean_percentile.low:.3f}, {mean_percentile.high:.3f}]")
print(f"Scipy mean BCa CI:           [{mean_bca.low:.3f}, {mean_bca.high:.3f}]")
print(f"Manual median percentile CI: [{median_manual[0]:.1f}, {median_manual[1]:.1f}]")
print(f"Scipy median percentile CI:  [{median_percentile.low:.1f}, {median_percentile.high:.1f}]")

if np.isnan(median_bca.low) or np.isnan(median_bca.high):
    print("Scipy median BCa CI:         not available because the sample median is tied and discrete")
else:
    print(f"Scipy median BCa CI:         [{median_bca.low:.1f}, {median_bca.high:.1f}]")


Post redesign respondent sample
Mean:   4.309
Median: 4.0

Manual mean percentile CI:   [4.294, 4.325]
Scipy mean percentile CI:    [4.294, 4.325]
Scipy mean BCa CI:           [4.293, 4.324]
Manual median percentile CI: [4.0, 4.0]
Scipy median percentile CI:  [4.0, 4.0]
Scipy median BCa CI:         not available because the sample median is tied and discrete


My hand bootstrap and scipy give almost the same percentile interval for the mean so that check looks good. The percentile and BCa mean intervals are also basically the same here and both are very tight around 4.31.

The median is 4 and its percentile interval is just 4 to 4. The BCa median interval is not available because the five point outcome has lots of ties so the jackknife values used by BCa do not move enough. That is a useful warning that a method can break down when the statistic is very discrete.


### Step 2.2: Coverage is a claim you can check (15 pts)

In [5]:
def response_for_sample(df, a, b, rng):
    z = (
        a
        + b * (df["s"].to_numpy() - 3.3)
        + 0.30 * df["engagement_z"].to_numpy()
        - 0.25 * df["had_support_ticket"].to_numpy()
    )
    p = 1 / (1 + np.exp(-z))
    return rng.random(len(df)) < p


coverage_rng = np.random.default_rng(2026)
n_reps = 100
invited_per_survey = 1000
B_coverage = 300

percentile_hits = 0
bca_hits = 0

for rep in range(n_reps):
    invited_idx = coverage_rng.choice(
        len(after), size=invited_per_survey, replace=False
    )
    invited = after.iloc[invited_idx]
    answered = response_for_sample(invited, -2.75, 2.00, coverage_rng)
    x = invited.loc[answered, "satisfaction"].to_numpy()

    boot_means = []
    for _ in range(B_coverage):
        boot_sample = coverage_rng.choice(x, size=len(x), replace=True)
        boot_means.append(boot_sample.mean())

    lo_pct, hi_pct = np.percentile(boot_means, [2.5, 97.5])
    percentile_hits += lo_pct <= true_after <= hi_pct

    bca_ci = stats.bootstrap(
        (x,),
        np.mean,
        n_resamples=B_coverage,
        method="BCa",
        random_state=rep,
        vectorized=True,
        batch=50,
    ).confidence_interval
    bca_hits += bca_ci.low <= true_after <= bca_ci.high

percentile_coverage = percentile_hits / n_reps
bca_coverage = bca_hits / n_reps

percentile_mc_se = np.sqrt(
    percentile_coverage * (1 - percentile_coverage) / n_reps
)
bca_mc_se = np.sqrt(bca_coverage * (1 - bca_coverage) / n_reps)

print(f"True post population mean: {true_after:.3f}")
print(f"Percentile coverage: {percentile_coverage:.3f} ± {percentile_mc_se:.3f}")
print(f"BCa coverage:        {bca_coverage:.3f} ± {bca_mc_se:.3f}")


True post population mean: 3.391
Percentile coverage: 0.000 ± 0.000
BCa coverage:        0.000 ± 0.000


Both interval methods had essentially zero coverage of the true post redesign population mean in this simulation. There is also no useful difference between their coverage rates because both fail in the same direction and the gap between them is not bigger than the Monte Carlo noise.

The main problem is not the interval formula. The sample itself is biased because happier users are much more likely to answer and the bootstrap only resamples the people who already answered. So it gives a tight interval around the respondent mean but that is not the same thing as the population mean.


## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [6]:
def cohens_d(x, y):
    n1 = len(x)
    n2 = len(y)
    var1 = np.var(x, ddof=1)
    var2 = np.var(y, ddof=1)
    pooled_sd = np.sqrt(
        ((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2)
    )
    return (np.mean(x) - np.mean(y)) / pooled_sd


welch = stats.ttest_ind(post_survey, pre_survey, equal_var=False)
d_observed = cohens_d(post_survey, pre_survey)

print(f"Observed difference: {observed_change:.3f} points")
print(f"Welch t statistic:   {welch.statistic:.3f}")
print(f"p value:             {welch.pvalue:.3e}")
print(f"Cohen's d:           {d_observed:.3f}")


Observed difference: 0.717 points
Welch t statistic:   66.331
p value:             0.000e+00
Cohen's d:           0.879


The pre and post respondent means are extremely statistically different and Cohen's d is large. I would still not say this proves the redesign caused a big increase in satisfaction because the response mechanism changed at the same time. The test is answering whether the two respondent samples have different means and they clearly do, but the real problem is whether those respondents represent the same population in the same way and they do not.


### Step 3.2: Power, before the fact

In [7]:
power_tool = TTestIndPower()

# Use the true simulated populations to get a reasonable SD in satisfaction points
var_before = before["satisfaction"].var(ddof=1)
var_after = after["satisfaction"].var(ddof=1)
population_sd = np.sqrt((var_before + var_after) / 2)

true_effect_points = 0.10
d_for_point_one = true_effect_points / population_sd

n_for_80 = power_tool.solve_power(
    effect_size=d_for_point_one,
    alpha=0.05,
    power=0.80,
    ratio=1.0,
    alternative="two-sided",
)

# Use the smaller respondent group as the sample size we know we can get
n_available = min(len(pre_survey), len(post_survey))
d_detectable = power_tool.solve_power(
    effect_size=None,
    nobs1=n_available,
    alpha=0.05,
    power=0.80,
    ratio=1.0,
    alternative="two-sided",
)
points_detectable = d_detectable * population_sd

print(f"Population SD: {population_sd:.3f}")
print(f"n needed per group for a 0.10 point effect: {np.ceil(n_for_80):,.0f}")
print(f"Available n per group used for MDE: {n_available:,}")
print(f"Detectable effect at that n: {points_detectable:.3f} points")


def simulate_one_gap(n, seed):
    local_rng = np.random.default_rng(seed)

    def make_period(base):
        engagement = local_rng.lognormal(3.5, 0.6, n)
        engagement_z = (engagement - engagement.mean()) / engagement.std()
        support = local_rng.random(n) < 0.22
        latent = (
            base
            + 0.35 * engagement_z
            - 0.60 * support
            + local_rng.normal(0, 0.75, n)
        )
        score = np.clip(np.rint(latent), 1, 5)
        return score, engagement_z, support, latent

    score_pre, z_pre, support_pre, latent_pre = make_period(BASE_BEFORE)
    score_post, z_post, support_post, latent_post = make_period(BASE_AFTER)

    p_pre = 1 / (
        1
        + np.exp(
            -(
                -1.0
                + 0.45 * (latent_pre - 3.3)
                + 0.30 * z_pre
                - 0.25 * support_pre
            )
        )
    )
    p_post = 1 / (
        1
        + np.exp(
            -(
                -2.75
                + 2.00 * (latent_post - 3.3)
                + 0.30 * z_post
                - 0.25 * support_post
            )
        )
    )

    r_pre = local_rng.random(n) < p_pre
    r_post = local_rng.random(n) < p_post

    survey_gap = score_post[r_post].mean() - score_pre[r_pre].mean()
    true_gap = score_post.mean() - score_pre.mean()
    return survey_gap, true_gap


rows = []
for n in [2000, 10000, 50000]:
    survey_gaps = []
    true_gaps = []
    for rep in range(100):
        survey_gap, true_gap = simulate_one_gap(n, seed=1000 + n + rep)
        survey_gaps.append(survey_gap)
        true_gaps.append(true_gap)

    rows.append({
        "users_per_period": n,
        "mean_observed_gap": np.mean(survey_gaps),
        "sd_observed_gap": np.std(survey_gaps, ddof=1),
        "mean_true_gap": np.mean(true_gaps),
    })

more_n_results = pd.DataFrame(rows)
display(more_n_results)


Population SD: 0.887
n needed per group for a 0.10 point effect: 1,237
Available n per group used for MDE: 7,638
Detectable effect at that n: 0.040 points


,users_per_period,mean_observed_gap,sd_observed_gap,mean_true_gap
0,2000,0.708,0.052,0.093
1,10000,0.716,0.023,0.098
2,50000,0.709,0.011,0.098


About 1,237 people per group would be enough for 80 percent power to detect a real 0.10 point change under a clean random sample. With the smaller respondent group we actually have, the design could detect a change of only about 0.04 points if sampling were unbiased.

But the simulation shows why power is not the main problem. As the population size goes from 2,000 to 50,000 the observed survey gap stays around 0.71 even though the true gap stays around 0.10. The random noise gets smaller but the bias stays, so more data makes the estimate more precise without making it more correct.


## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [8]:
# Weight the post redesign respondents using only the two observed covariates
post_for_weighting = after.copy()
post_for_weighting["responded"] = responded_after

X = post_for_weighting[["engagement_z", "had_support_ticket"]].astype(float)
y = post_for_weighting["responded"].astype(int)

response_model = LogisticRegression(max_iter=1000)
response_model.fit(X, y)

post_for_weighting["p_hat"] = response_model.predict_proba(X)[:, 1]
respondents = post_for_weighting[post_for_weighting["responded"]].copy()
respondents["weight"] = 1 / respondents["p_hat"]

weighted_post_mean = np.average(
    respondents["satisfaction"],
    weights=respondents["weight"],
)

# Simple bootstrap of the weighted respondent rows
weight_rng = np.random.default_rng(123)
weighted_boot = []
for _ in range(2000):
    idx = weight_rng.integers(0, len(respondents), size=len(respondents))
    sample = respondents.iloc[idx]
    weighted_boot.append(
        np.average(sample["satisfaction"], weights=sample["weight"])
    )

weighted_ci = np.percentile(weighted_boot, [2.5, 97.5])

print(f"True post mean:           {true_after:.3f}")
print(f"Raw respondent mean:      {observed_after:.3f}")
print(f"IPW corrected mean:       {weighted_post_mean:.3f}")
print(f"IPW bootstrap 95% CI:     [{weighted_ci[0]:.3f}, {weighted_ci[1]:.3f}]")


True post mean:           3.391
Raw respondent mean:      4.309
IPW corrected mean:       4.137
IPW bootstrap 95% CI:     [4.116, 4.159]


**My revision**

The first prompt did not say clearly which period I was correcting and it used engagement_minutes even though my response mechanism was built with engagement_z. In version 2 I said to correct the post redesign mean only and I gave the exact dataframe, response variable, outcome, covariates and seed.

**My check**

The true post redesign mean is about 3.391 and the raw respondent mean is about 4.309. The weighting moves the estimate down to about 4.137 but that is still far above the truth so the correction did not work.

**What the weights are doing in my own words**

The logistic model estimates each user's chance of answering from engagement and support tickets. For people who actually answered I take 1 divided by that estimated chance, so respondents who looked less likely to answer count more in the weighted average. That can balance the measured covariates but it cannot fully fix this case because satisfaction itself also changes response probability.

**One specific problem I noticed**

The bootstrap code keeps the fitted response probabilities fixed instead of fitting the response model again inside every bootstrap sample, so the interval does not include uncertainty from estimating the response model. More importantly the response model leaves out the direct satisfaction term that actually drives nonresponse, so the main identification assumption is broken.

**Would I use AI for this step again**

Yes, I would use it again for a quick first version because the weighting setup was useful, but I would still check the assumptions and compare the result with a known truth before trusting it.


### Task 4.2: The slide (5 pts)

### Slide: Satisfaction rose among survey respondents, but the survey cannot prove the redesign caused it

Post redesign respondents averaged **4.31 out of 5**, with a 95 percent BCa interval of about **4.29 to 4.32**. The survey clearly shows that the people who answered after the redesign reported higher satisfaction than the people who answered before it.

But this does not show that satisfaction increased by the same amount for all users. The response rate fell from about 27.5 percent to 15.3 percent and the new response mechanism made high satisfaction users much more likely to answer. In the simulation the true population improvement is only about 0.10 points even though the observed survey gap is about 0.72 points. The weighting attempt using engagement and support tickets still misses the true mean because response also depends directly on satisfaction.

The clean fix is to survey a random sample of users whether or not they finish onboarding and use the same follow up process before and after the redesign.


### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* GPT-5.6 Sol, October 26, 2026

**Version 1 prompt:**

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

**Reply:**

You can fit a logistic regression for the response indicator using the observed covariates, predict each user's response probability, and give each respondent a weight equal to 1 divided by that probability.

```python
from sklearn.linear_model import LogisticRegression
import numpy as np

X = df[["engagement_minutes", "had_support_ticket"]].astype(float)
y = df["responded"].astype(int)

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

df["p_hat"] = model.predict_proba(X)[:, 1]
respondents = df[df["responded"]].copy()
respondents["weight"] = 1 / respondents["p_hat"]

weighted_mean = np.average(
    respondents["satisfaction"],
    weights=respondents["weight"]
)

rng = np.random.default_rng(42)
boot = []
for _ in range(2000):
    idx = rng.integers(0, len(respondents), len(respondents))
    sample = respondents.iloc[idx]
    boot.append(np.average(sample["satisfaction"], weights=sample["weight"]))

lo, hi = np.percentile(boot, [2.5, 97.5])
print(weighted_mean)
print(lo, hi)
```

The key assumption is that response is independent of satisfaction after conditioning on the covariates in the response model. This is a missing at random type assumption. If response still depends directly on satisfaction after conditioning on engagement and support tickets, the model does not have enough information to remove the selection bias.

Given your mechanism, no, this correction should not fully recover the true mean because satisfaction itself affects response and satisfaction is not included in the response model.

**Change I made:** Version 1 did not identify the exact period or the exact dataframe and response variable, and it used engagement_minutes instead of the standardized engagement_z used in my simulation. I fixed those details in version 2 so the code matched my notebook.

**Version 2 prompt:**

Act as a survey statistician working on nonresponse adjustment. I am correcting the POST redesign survey mean only.

My post redesign DataFrame is named after. The outcome is satisfaction. The binary response indicator is responded. The two covariates are engagement_z and had_support_ticket. engagement_z is already standardized. Use LogisticRegression on the full simulated post population to model responded from those two covariates only. For respondents, use inverse predicted response probability as the weight and return the weighted post redesign satisfaction mean with a percentile bootstrap interval using seed 123.

Do not add satisfaction or latent s to the response model because I want to test whether weighting on the observed covariates can fix the bias. Explain the assumption needed for this correction and explain what happens because my real response mechanism directly includes latent satisfaction s. Keep the code simple and use my exact variable names.

**Reply:**

Use the full post redesign population to fit the response model, then calculate weights only for the people who responded.

```python
from sklearn.linear_model import LogisticRegression
import numpy as np

after = after.copy()
after["responded"] = responded_after

X = after[["engagement_z", "had_support_ticket"]].astype(float)
y = after["responded"].astype(int)

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

after["p_hat"] = model.predict_proba(X)[:, 1]
respondents = after[after["responded"]].copy()
respondents["weight"] = 1 / respondents["p_hat"]

weighted_mean = np.average(
    respondents["satisfaction"],
    weights=respondents["weight"]
)

rng = np.random.default_rng(123)
boot = []
for _ in range(2000):
    idx = rng.integers(0, len(respondents), len(respondents))
    sample = respondents.iloc[idx]
    boot.append(
        np.average(sample["satisfaction"], weights=sample["weight"])
    )

lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Weighted post mean: {weighted_mean:.3f}")
print(f"95% bootstrap CI: [{lo:.3f}, {hi:.3f}]")
```

The correction needs response to be independent of satisfaction after conditioning on engagement_z and had_support_ticket, plus positive response probability for the kinds of users we want to represent. In your simulated mechanism that assumption fails because latent satisfaction s directly changes the chance of responding even after the two covariates are held fixed.

So the answer is no. The weighting can reduce some bias that comes through engagement and support tickets, but it cannot remove the remaining selection on satisfaction itself. You should compare the weighted estimate with the known post population mean and report the failure honestly.

**Change after version 2:** I kept the weighting idea but checked the corrected mean against the known true post mean. I also noted that the simple bootstrap keeps the fitted weights fixed, so it misses some model fitting uncertainty.


## Before you submit

[x] Every heading above is still here, in order, with work under it.

[x] Every number I report is produced by a cell above it and every claim has a text cell.

[x] The verification checkpoint is met before anything built on it.

[x] Phase 4 includes both prompt versions and both replies in the transcript.

[ ] Run Runtime > Restart and run all once in Colab and make sure there are no red errors.

[ ] Set the Colab share link to Anyone with the link and submit that link on Canvas.

[ ] Save the notebook to GitHub at `Problem Set 2/Econ_3916_PS2.ipynb` and add a short `README.md` in the same folder.
